
# Databricks markdown cell
# 📘 Broadcast Join in Apache Spark (Databricks)

---

## 🔷 What is a Broadcast Join?

A **broadcast join** is a performance optimization technique in Apache Spark to efficiently join a **large dataset with a small dataset**.

- Spark copies the small table to all executor nodes.
- The join is performed locally on each node.

👉 This avoids expensive network shuffle and improves performance.

---

## 🔷 When do we use Broadcast Join?

Broadcast join is used when:

- One dataset is **small enough to fit in memory**
- The other dataset is **very large**
- We want to **avoid shuffle operations**
- We are working in distributed systems like **Databricks / Spark**

---

### 📏 Default Threshold

- `spark.sql.autoBroadcastJoinThreshold`
- Default value: ~10MB (configurable)
- If table size is below this, Spark may auto-broadcast it

---

## 🔷 How Broadcast Join Works

- Spark identifies the smaller dataset.
- The small dataset is broadcast to all executor nodes.
- Each executor stores it in memory.
- The large dataset is processed in partitions.
- Join happens locally on each node (no shuffle for small table).

---

## 🔷 Real-Time Use Cases

**🛒 E-commerce: Orders + Products**
- Large table: orders
- Small table: products
- Broadcast: products

**🏦 Banking: Transactions + Branch Info**
- Large table: transactions
- Small table: branch_master
- Broadcast: branch_master

**📢 AdTech: Click Logs + Campaign Data**
- Large table: click logs
- Small table: campaign metadata

**🏥 Healthcare Systems**
- Large: patient visit records
- Small: disease codes / reference lookup tables

**📊 Streaming Use Case (Kafka + Spark)**
- Large: streaming events (clicks, logs)
- Small: user profile table

---

## 🔷 PySpark Example

python
from pyspark.sql.functions import broadcast

result_df = orders.join(broadcast(products), "product_id")


---

## 🔷 Spark Execution Behavior

**Without Broadcast Join:**
- Full shuffle across cluster
- High network I/O
- Slower execution

**With Broadcast Join:**
- Small table replicated to all nodes
- No shuffle for small table
- Faster execution

---

## 🔷 Advantages

- 🚀 Faster joins (avoids shuffle)
- 📉 Reduces network I/O
- ⚡ Improves performance in distributed processing
- 🧠 Efficient for star-schema workloads

---

## 🔷 Disadvantages

- ❌ Large “small table” can cause memory overflow
- ❌ Not suitable for large-to-large joins
- ❌ Requires proper threshold tuning

---

## 🔷 Summary

| Join Type      | When to Use                |
|----------------|---------------------------|
| Shuffle Join   | Both datasets are large    |
| Broadcast Join | One small + one large set  |

---

## 🔷 Key Takeaway

Broadcast join is one of the most powerful Spark optimizations in Databricks when used correctly. It significantly reduces shuffle cost and improves performance, especially in ETL pipelines where dimension tables are small and fact tables are large.

# 📘 Broadcast Join Example with Two DataFrames (PySpark)

This example demonstrates how a **broadcast join** works using two DataFrames in Spark.

---

## 🔷 Step 1: Create Sample DataFrames

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

spark = SparkSession.builder.getOrCreate()

# Large dataset (Fact table)
orders_data = [
    (1, 101, 500),
    (2, 102, 300),
    (3, 101, 700),
    (4, 103, 200),
    (5, 104, 900)
]

orders_df = spark.createDataFrame(orders_data, ["order_id", "product_id", "amount"])

# Small dataset (Dimension table)
products_data = [
    (101, "Laptop"),
    (102, "Phone"),
    (103, "Tablet"),
    (104, "Headphones")
]

products_df = spark.createDataFrame(products_data, ["product_id", "product_name"])

In [0]:
🔷 Step 2: Normal Join (Shuffle Join)
normal_join_df = orders_df.join(products_df, "product_id", "inner")
normal_join_df.show()

Step 3: Broadcast Join (Optimized)
broadcast_join_df = orders_df.join(
    broadcast(products_df),
    "product_id",
    "inner"
)

broadcast_join_df.show()

products_df is broadcast to all executor nodes, avoiding shuffle.

🔷 Step 4: Output Result
+----------+----------+------+-------------+
|product_id| order_id |amount|product_name |
+----------+----------+------+-------------+
|   101    |    1     | 500  | Laptop      |
|   102    |    2     | 300  | Phone       |
|   101    |    3     | 700  | Laptop      |
|   103    |    4     | 200  | Tablet      |
|   104    |    5     | 900  | Headphones  |
+----------+----------+------+-------------+

🔷 Key Takeaway
orders_df = Large dataset (Fact table)
products_df = Small dataset (Dimension table)
Broadcasting the small table avoids shuffle and improves performance significantly
